In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from itertools import product

import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-darkgrid')


from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import joblib

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller

print("✅ Libraries loaded")

✅ Libraries loaded


In [2]:
# Load dataset and identify key columns
DATASET = r'feature_engineered_finance_dataset.csv'
df_raw = pd.read_csv(DATASET)

date_col = None
expense_col = None
cat_col = None
for col in df_raw.columns:
    low = col.lower()
    if date_col is None and ('date' in low or 'tanggal' in low):
        date_col = col
    if expense_col is None and ('total_harga' in low or 'expense' in low):
        expense_col = col
    if cat_col is None and ('kategori' in low or 'category' in low):
        cat_col = col
df_raw[date_col] = pd.to_datetime(df_raw[date_col])
df_raw = df_raw.dropna(subset=[date_col])
print(f"✅ Data loaded: {df_raw.shape[0]} rows")

✅ Data loaded: 6573 rows


In [3]:
# Aggregate daily and monthly data
daily = df_raw.groupby(date_col)[expense_col].sum().reset_index()
daily.columns = ['date', 'daily_expense']
daily = daily.set_index('date').asfreq('D', fill_value=0).reset_index()

monthly = df_raw.groupby(df_raw[date_col].dt.to_period('M'))[expense_col].sum().reset_index()
monthly.columns = ['month', 'total_expense']
monthly['month'] = monthly['month'].astype(str)
print(f"✅ Daily ({len(daily)} days) and Monthly ({len(monthly)} months) aggregations done.")

✅ Daily (731 days) and Monthly (24 months) aggregations done.


In [4]:
# Create daily features for Deep Learning model
def create_features(df_daily, df_raw):
    feats = df_daily.copy()
    for lag in [1,2,3,7,30]:
        feats[f'lag_{lag}'] = feats['daily_expense'].shift(lag)
    for w in [7,30]:
        feats[f'rolling_mean_{w}'] = feats['daily_expense'].rolling(w, min_periods=1).mean()
    feats['day_of_week'] = feats['date'].dt.dayofweek
    feats['month'] = feats['date'].dt.month
    feats['day_of_month'] = feats['date'].dt.day
    feats['is_weekend'] = (feats['day_of_week'] >= 5).astype(int)
    feats['days_in_month'] = feats['date'].dt.days_in_month
    feats['mtd_progress'] = (feats['day_of_month'] - 1) / (feats['days_in_month'] - 1)
    feats['mtd_progress'] = feats['mtd_progress'].clip(0,1)
    txn = df_raw.groupby(df_raw[date_col].dt.date).size().reset_index()
    txn.columns = ['date', 'transaction_count']
    txn['date'] = pd.to_datetime(txn['date'])
    feats = feats.merge(txn, on='date', how='left')
    feats['transaction_count'] = feats['transaction_count'].fillna(0)
    return feats

feature_df = create_features(daily, df_raw)
print(f"✅ Features created. Shape: {feature_df.shape}")

✅ Features created. Shape: (731, 16)


In [5]:
# Train ARIMA model and get forecast
monthly_vals = monthly['total_expense'].values
best_aic = np.inf
best_order = None
for p,d,q in product(range(3), range(2), range(3)):
    if p==0 and q==0: continue
    try:
        # Removed disp=False as it is deprecated/removed in newer statsmodels versions
        model = ARIMA(monthly_vals, order=(p,d,q)).fit()
        if model.aic < best_aic:
            best_aic, best_order = model.aic, (p,d,q)
    except:
        continue

# Check if best_order was found, if not, assign a default
if best_order is None:
    best_order = (1, 1, 1) # Assign a common default ARIMA order

arima_model = ARIMA(monthly_vals, order=best_order).fit()
arima_forecast = arima_model.forecast(steps=2)
print(f"✅ ARIMA model trained. Forecast: Rp {arima_forecast[0]:,.0f}")

✅ ARIMA model trained. Forecast: Rp 27,178,886


In [6]:
# Train SARIMAX model and get forecast
monthly_exog = daily.copy()
monthly_exog['month'] = daily['date'].dt.to_period('M')
monthly_exog = monthly_exog.groupby('month').agg({
    'daily_expense': 'sum',
    'date': 'count'
}).reset_index()
monthly_exog.columns = ['month', 'total_expense', 'days_in_month']

txn_monthly = df_raw.groupby(df_raw[date_col].dt.to_period('M')).size().reset_index()
txn_monthly.columns = ['month', 'transaction_count']
monthly_exog = monthly_exog.merge(txn_monthly, on='month', how='left').fillna(0)
monthly_exog['month'] = monthly_exog['month'].astype(str)

scaler_exog = StandardScaler()
exog_scaled = scaler_exog.fit_transform(monthly_exog[['days_in_month','transaction_count']])

sarimax_model = SARIMAX(monthly_exog['total_expense'], exog=exog_scaled,
                        order=(1,1,1), seasonal_order=(1,1,1,12),
                        enforce_stationarity=False, enforce_invertibility=False,
                        initialization='approximate_diffuse')
sarimax_fitted = sarimax_model.fit(disp=False)

last_exog_scaled = exog_scaled[-1]
future_exog = np.array([[last_exog_scaled[0], last_exog_scaled[1]*1.05] for _ in range(2)])
sarimax_forecast = sarimax_fitted.forecast(steps=2, exog=future_exog)
sarimax_forecast = np.maximum(sarimax_forecast, 0)
print(f"✅ SARIMAX model trained. Forecast: Rp {sarimax_forecast.iloc[0]:,.0f}")

✅ SARIMAX model trained. Forecast: Rp 889,713


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [7]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Filter for active transaction days to help the model learn expense patterns better
active_days = feature_df[feature_df['daily_expense'] > 0].copy()

dl_feats = ['lag_1','lag_2','lag_3','rolling_mean_7','rolling_mean_30',
            'day_of_week','month','is_weekend','mtd_progress','transaction_count']

available = [f for f in dl_feats if f in feature_df.columns]

# We will use the full feature_df but add a weight to days with expenses
X = feature_df[available].fillna(0).values
y = feature_df['daily_expense'].values

# Sample weights: Give more importance to non-zero expense days so the model doesn't just learn to predict 0
sample_weights = np.where(y > 0, 5.0, 1.0)

# Split data
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
weights_train = sample_weights[:split]

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1,1)).flatten()
y_test_scaled = scaler_y.transform(y_test.reshape(-1,1)).flatten()

print(f"✅ Model dataset prepared with increased weight on active spending days ({len(active_days)} days found).")

✅ Model dataset prepared with increased weight on active spending days (5 days found).


In [8]:
# Build and compile Deep Learning model
import sys
import subprocess

# Fix for TensorFlow ImportError in specific environments
try:
    import tensorflow as tf
except ImportError:
    print("⚠️ Fixing TensorFlow installation issues...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "tensorflow", "--quiet"])
    import tensorflow as tf

from tensorflow.keras import layers, Model, Input

def build_dl_model(input_dim):
    inputs = Input(shape=(input_dim,))
    x = layers.Dense(128, activation='relu')(inputs)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1)(x)
    model = Model(inputs, outputs)
    model.compile(
        optimizer='adam',
        loss='huber',
        metrics=['mae']
    )
    return model

dl_model = build_dl_model(X_train_scaled.shape[1])
print("✅ Deep Learning model built successfully.")

✅ Deep Learning model built successfully.


In [9]:
import os
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Create the models directory if it doesn't exist to prevent FileNotFoundError
os.makedirs('models', exist_ok=True)

# Training with sample weights to prioritize learning from used data patterns
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)

history = dl_model.fit(
    X_train_scaled, y_train_scaled,
    sample_weight=weights_train,
    validation_data=(X_test_scaled, y_test_scaled),
    epochs=100, batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=0
)

# Re-save the improved model
dl_model.save('models/forecast_model.keras')
print(f"✅ Improved model trained and saved in 'models/'. Best val loss: {min(history.history['val_loss']):.6f}")

✅ Improved model trained and saved in 'models/'. Best val loss: 0.000018


In [10]:
def recursive_forecast_monthly(start_features, remaining_days, model, scaler_X, scaler_y, feature_names, max_val=10000000):
    try:
        current = start_features.copy()
        daily_preds = []
        # Use historical max as a safety cap if not provided
        for day in range(remaining_days):
            scaled = scaler_X.transform(current)
            pred_scaled = model.predict(scaled, verbose=0)
            pred = scaler_y.inverse_transform(pred_scaled).flatten()[0]

            # Safety checks: prevent NaN, negative, or explosive growth
            if np.isnan(pred) or np.isinf(pred): pred = 0
            pred = max(0, min(pred, max_val))

            daily_preds.append(pred)

            # Update features for next day
            new_row = current[0].copy()
            if 'lag_3' in feature_names: new_row[feature_names.index('lag_3')] = new_row[feature_names.index('lag_2')]
            if 'lag_2' in feature_names: new_row[feature_names.index('lag_2')] = new_row[feature_names.index('lag_1')]
            if 'lag_1' in feature_names: new_row[feature_names.index('lag_1')] = pred

            current = new_row.reshape(1, -1)

        return sum(daily_preds), True, "Success", daily_preds
    except Exception as e:
        return 0, False, str(e), []
print("✅ Fixed Recursive forecasting function with safety caps.")

✅ Fixed Recursive forecasting function with safety caps.


In [11]:
# Calculate Ensemble Forecast
last_features = feature_df.iloc[-1][available].values.reshape(1, -1)
today = datetime.now()
days_left = (today.replace(day=1, month=today.month%12+1) - today).days
if days_left <= 0:
    days_left = 30

dl_total, dl_success, dl_msg, _ = recursive_forecast_monthly(
    last_features, days_left, dl_model, scaler_X, scaler_y, available
)

arima_next = arima_forecast[0]
sarimax_next_val = sarimax_forecast.iloc[0]

# WEIGHTED ENSEMBLE (Prioritizing ARIMA for accuracy)
if dl_total > 0:
    ensemble_forecast = (0.8 * arima_next) + (0.1 * sarimax_next_val) + (0.1 * dl_total)
else:
    ensemble_forecast = (0.9 * arima_next) + (0.1 * sarimax_next_val)

print(f"✅ Ensemble Forecast (Weighted): Rp {ensemble_forecast:,.0f}")

✅ Ensemble Forecast (Weighted): Rp 21,832,842


In [12]:
import os

# Create the models directory if it doesn't exist
os.makedirs('models', exist_ok=True)
print("✅ 'models' directory created.")

✅ 'models' directory created.


In [13]:
# Save models and scalers
dl_model.save('models/forecast_model.keras')
joblib.dump(scaler_X, 'models/scaler_X.save')
joblib.dump(scaler_y, 'models/scaler_y.save')
joblib.dump(scaler_exog, 'models/scaler_exog.save')
print("✅ Models and scalers saved.")

✅ Models and scalers saved.


### 📅 Prediksi Khusus: Juni
Sel ini menghitung estimasi pengeluaran total untuk bulan Juni menggunakan metode Ensemble.

In [14]:
import pandas as pd
import numpy as np
from datetime import datetime

# 1. Tentukan target bulan (Juni tahun berjalan atau tahun depan sesuai data)
target_month = 6
target_year = 2025  # Target: Juni 2025

# 2. Hitung jumlah hari di bulan Juni
import calendar
days_in_june = calendar.monthrange(target_year, target_month)[1]

# 3. Ambil fitur terakhir dari data historis sebagai titik awal
last_features_june = feature_df.iloc[-1][available].values.reshape(1, -1)

# 4. Jalankan Prediksi Deep Learning (Recursive)
june_dl_total, success, msg, june_daily_list = recursive_forecast_monthly(
    last_features_june, days_in_june, dl_model, scaler_X, scaler_y, available
)

# 5. Ambil Prediksi ARIMA & SARIMAX (Bulan depan)
# (Menggunakan hasil forecast langkah 1 dari model yang sudah dilatih)
june_arima = arima_forecast[0]
june_sarimax = sarimax_forecast.iloc[0]

# 6. Ensemble Weighting
# Kita beri bobot lebih tinggi ke ARIMA karena kestabilannya pada data bulanan
june_ensemble = (0.7 * june_arima) + (0.2 * june_dl_total) + (0.1 * june_sarimax)

print("="*50)
print(f"PREDIKSI PENGELUARAN BULAN JUNI {target_year}")
print(f"Total Estimasi: Rp {june_ensemble:,.0f}")
print("="*50)
print(f"Rincian Kontribusi:")
print(f"- ARIMA         : Rp {june_arima:,.0f}")
print(f"- Deep Learning : Rp {june_dl_total:,.0f}")
print(f"- SARIMAX       : Rp {june_sarimax:,.0f}")

PREDIKSI PENGELUARAN BULAN JUNI 2025
Total Estimasi: Rp 19,115,885
Rincian Kontribusi:
- ARIMA         : Rp 27,178,886
- Deep Learning : Rp 8,466
- SARIMAX       : Rp 889,713


In [15]:
# Calculate Confidence Interval
def smape(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

train_monthly = monthly_vals[:-3]
test_monthly = monthly_vals[-3:]
# Removed disp=False as it is unsupported in newer statsmodels versions
arima_eval = ARIMA(train_monthly, order=best_order).fit()
pred_arima = arima_eval.forecast(3)
smape_arima = smape(test_monthly, pred_arima)

y_pred_daily = dl_model.predict(X_test_scaled, verbose=0).flatten()
y_true_daily = y_test
smape_daily = smape(y_true_daily, y_pred_daily)

errors = monthly_vals - arima_model.fittedvalues
std_err = np.std(errors)
ci_margin = 1.96 * std_err
lower = ensemble_forecast - ci_margin
upper = ensemble_forecast + ci_margin
print(f"✅ 95% Confidence Interval: Rp {lower:,.0f} — Rp {upper:,.0f}")

✅ 95% Confidence Interval: Rp 5,825,211 — Rp 37,840,473


In [26]:
import json

# Save the ensemble forecast and confidence interval to a JSON file
forecast_results = {
    "forecast_this_month": float(ensemble_forecast),
    "confidence_lower": float(lower),
    "confidence_upper": float(upper)
}

with open('models/ensemble_forecast_results.json', 'w') as f:
    json.dump(forecast_results, f)

print("✅ Ensemble forecast results saved to 'models/ensemble_forecast_results.json'.")

✅ Ensemble forecast results saved to 'models/ensemble_forecast_results.json'.


In [27]:
%%writefile main.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import numpy as np
import joblib
import tensorflow as tf
import json
import os

app = FastAPI(title="Finance Forecast API")

# Use absolute paths for Railway reliability
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_PATH = os.path.join(BASE_DIR, 'models', 'forecast_model.keras')
SCALER_X_PATH = os.path.join(BASE_DIR, 'models', 'scaler_X.save')
SCALER_Y_PATH = os.path.join(BASE_DIR, 'models', 'scaler_y.save')
JSON_PATH = os.path.join(BASE_DIR, 'models', 'ensemble_forecast_results.json')

# Load Model & Scalers
MODEL = tf.keras.models.load_model(MODEL_PATH)
SCALER_X = joblib.load(SCALER_X_PATH)
SCALER_Y = joblib.load(SCALER_Y_PATH)

with open(JSON_PATH, 'r') as f:
    ENSEMBLE_FORECAST_RESULTS = json.load(f)

class PredictRequest(BaseModel):
    lag_1: float
    lag_2: float
    lag_3: float
    rolling_mean_7: float
    rolling_mean_30: float
    day_of_week: int
    month: int
    is_weekend: int
    mtd_progress: float
    transaction_count: int

@app.get("/")
def home():
    return {"message": "Finance Forecast API is running"}

@app.get("/health")
def health():
    return {"status": "healthy"}

@app.post("/predict")
def predict(data: PredictRequest):
    try:
        return {
            "success": True,
            "message": "OK",
            "forecast_this_month": ENSEMBLE_FORECAST_RESULTS["forecast_this_month"],
            "confidence_lower": ENSEMBLE_FORECAST_RESULTS["confidence_lower"],
            "confidence_upper": ENSEMBLE_FORECAST_RESULTS["confidence_upper"]
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


Overwriting main.py


In [28]:
%%bash --bg
uvicorn main:app --host 0.0.0.0 --port 8000 &> server.log

In [29]:
!python test_api.py

Testing API at http://localhost:8000/predict...
⚠️ Attempt 1 failed: 500 Server Error: Internal Server Error for url: http://localhost:8000/predict
⚠️ Attempt 2 failed: 500 Server Error: Internal Server Error for url: http://localhost:8000/predict
⚠️ Attempt 3 failed: 500 Server Error: Internal Server Error for url: http://localhost:8000/predict
❌ All attempts to connect to the server failed. Check server.log for errors.


In [31]:
!cat server.log

2026-06-04 09:29:21.855217: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1780565361.856529    9895 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13273 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
INFO:     Started server process [9895]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
ININFO:     127.0.0.1:51190 - "POST /predict HTTP/1.1" 500 Internal Server Error
INFO:     127.0.0.1:51204 - "POST /predict HTTP/1.1" 500 Internal Server Error


In [32]:
%%bash
fuser -k 8000/tcp

  9155

8000/tcp:           


In [33]:
%%bash --bg
uvicorn main:app --host 0.0.0.0 --port 8000 &> server.log

In [34]:
!python test_api.py

Testing API at http://localhost:8000/predict...
⚠️ Attempt 1 failed: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /predict (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7bf043ca2180>: Failed to establish a new connection: [Errno 111] Connection refused'))
✅ Status Code: 200
✅ Response JSON: {
  "success": true,
  "message": "OK",
  "forecast_this_month": 21832842.45147517,
  "confidence_lower": 5825211.498026136,
  "confidence_upper": 37840473.4049242
}


In [30]:
!cat server.log

2026-06-04 09:29:21.855217: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1780565361.856529    9895 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13273 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
INFO:     Started server process [9895]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
ININFO:     127.0.0.1:51190 - "POST /predict HTTP/1.1" 500 Internal Server Error
INFO:     127.0.0.1:51204 - "POST /predict HTTP/1.1" 500 Internal Server Error


In [25]:
%%writefile test_api.py
import requests
import json
import time

# In Colab, localhost or 0.0.0.0 is often more reliable than 127.0.0.1 for internal requests
url = "http://localhost:8000/predict"

payload = {
    "lag_1": 50000.0,
    "lag_2": 45000.0,
    "lag_3": 60000.0,
    "rolling_mean_7": 52000.0,
    "rolling_mean_30": 50000.0,
    "day_of_week": 2,
    "month": 5,
    "is_weekend": 0,
    "mtd_progress": 0.5,
    "transaction_count": 3
}

print(f"Testing API at {url}...")

# Add an initial delay to allow the server to start up fully
time.sleep(5)

for i in range(3):
    try:
        response = requests.post(url, json=payload, timeout=5)
        response.raise_for_status()
        print("✅ Status Code:", response.status_code)
        response_json = response.json()
        print("✅ Response JSON:", json.dumps(response_json, indent=2))
        break
    except Exception as e:
        print(f"⚠️ Attempt {i+1} failed: {e}")
        if i < 2:
            time.sleep(2)
        else:
            print("❌ All attempts to connect to the server failed. Check server.log for errors.")


Overwriting test_api.py


In [19]:
%%writefile requirements.txt
fastapi
uvicorn[standard]
pydantic
numpy
joblib
scikit-learn
tensorflow-cpu
pandas
python-multipart

Writing requirements.txt


In [35]:
%%writefile Procfile
web: uvicorn main:app --host 0.0.0.0 --port ${PORT:-8000}

Writing Procfile


In [21]:
%%bash --bg
uvicorn main:app --host 0.0.0.0 --port 8000 &> server.log